In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations

In [2]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * np.std(a, ddof=1)**2 + (nb - 1) * np.std(b, ddof=1)**2) / (na + nb - 2))
    if pooled_std == 0:
        return 0.0
    return (np.mean(a) - np.mean(b)) / pooled_std

def mean_diff(a, b):
    """Calculate mean differences between two groups."""

    return (np.mean(a) - np.mean(b))

def run_ttest(vals_a, vals_b):
    if len(vals_a) < 2 or len(vals_b) < 2:
        return np.nan, np.nan
    _, p_value = stats.ttest_ind(vals_a, vals_b, equal_var=False)
    effect_size = cohens_d(vals_a, vals_b)
    return p_value, effect_size

import numpy as np

def run_permutation_test(vals_a, vals_b, n_permutations=10000, random_state=None):
    vals_a = np.asarray(vals_a)
    vals_b = np.asarray(vals_b)

    if len(vals_a) < 2 or len(vals_b) < 2:
        return np.nan, np.nan

    observed_stat = np.mean(vals_a) - np.mean(vals_b)
    combined = np.concatenate([vals_a, vals_b])
    n_a = len(vals_a)

    rng = np.random.default_rng(random_state)

    perm_stats = np.empty(n_permutations)
    for i in range(n_permutations):
        permuted = rng.permutation(combined)
        perm_a = permuted[:n_a]
        perm_b = permuted[n_a:]
        perm_stats[i] = np.mean(perm_a) - np.mean(perm_b)

    p_value = (np.sum(np.abs(perm_stats) >= np.abs(observed_stat)) + 1) / (n_permutations + 1)
    # effect_size = cohens_d(vals_a, vals_b)
    effect_size = mean_diff(vals_a, vals_b)

    return p_value, effect_size

In [3]:
merged_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/all_metrics_1and2percent.csv')
merged_df = merged_df[['disease', 'setting', 'method', 'category',
                        'bio_sim_300', 'bio_sim_150',
                        'top_recall_150', 'succ_150',
                        'top_recall_300', 'succ_300',
                        'bedroc_150', 'bedroc_300', 'auroc']]

selected_metrics = ['bio_sim_300', 'bio_sim_150',
                    'top_recall_150', 'top_recall_300',
                    'bedroc_150', 'bedroc_300', 'auroc']

## OC

In [4]:
merged_df['setting'].unique(),merged_df['method'].unique()

(array(['df_gnn_occsvm_pred', '2019_occ_deep_svd_pred/pred.pkl',
        '2019_nn_all_pred', '2019_rf_renamed_pred', '2019_mf_add_pred',
        'graphsage', 'gcn'], dtype=object),
 array(['early_fused', 'early_fused_ppi', 'late_fused', 'late_fused_ppi',
        'ppi_emb_n2v', 'ppi_emb_dw', 'ppi_emb_df', 'literature_emb',
        'seq_emb_esm', 'seq_emb_port', 'mid_linear_fused', 'mid_geo_fused',
        'mid_linear_fused_ppi', 'mid_geo_fused_ppi', 'occ_svm',
        'uniport_ppi_2019', 'mid_fused', 'mid_fused_ppi'], dtype=object))

In [5]:
selected_pairs = [('df_gnn_occsvm_pred', 'occ_svm'),
    ('df_gnn_occsvm_pred', 'ppi_emb_n2v'),
    ('2019_nn_all_pred', 'ppi_emb_n2v'),
    ('2019_occ_deep_svd_pred/pred.pkl', 'uniport_ppi_2019')]
subresults = (merged_df.set_index(['setting', 'method']).loc[selected_pairs].reset_index())

method_map = {
    ('df_gnn_occsvm_pred', 'occ_svm'): 'occ-svm',
    ('df_gnn_occsvm_pred', 'ppi_emb_n2v'): 'bagging-svm',
    ('2019_nn_all_pred', 'ppi_emb_n2v'): 'bagging-dnn',
    ('2019_occ_deep_svd_pred/pred.pkl', 'uniport_ppi_2019'): 'occ-deepsvdd',
}

# (optional but recommended) normalize strings to avoid hidden whitespace issues
subresults['setting'] = subresults['setting'].astype(str).str.strip()
subresults['method']  = subresults['method'].astype(str).str.strip()

# build a MultiIndex key per row and map using the dict
keys = pd.MultiIndex.from_frame(subresults[['setting', 'method']])
new_method = keys.map(method_map)  # returns array with mapped values or NaN

# apply mapped names where available
subresults.loc[~pd.isna(new_method), 'method'] = new_method[~pd.isna(new_method)]

subresults = subresults.sort_values("category").reset_index(drop=True)



In [24]:
avg_df = (
    subresults
    .groupby(["method", "category"], as_index=False)[selected_metrics]
    .mean()
)
avg_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/mean_occvsbag_category.csv')

In [7]:
results = []

for pair in [['bagging-svm', 'occ-svm'], ['bagging-dnn', 'occ-deepsvdd']]:
    for metric in selected_metrics:
        vals_a = subresults[subresults['method'] == pair[0]][metric].dropna().values
        vals_b = subresults[subresults['method'] == pair[1]][metric].dropna().values

        p_value, effect_size = run_permutation_test(vals_a, vals_b)

        results.append({
            'setting': 'oc vs bagging',
            'metric': metric,
            'compared_groups': f'{pair[0]} vs {pair[1]}',
            'p_value': p_value,
            'observed_difference': effect_size
        })

results_oc = pd.DataFrame(results)

In [8]:
results_oc

,setting,metric,compared_groups,p_value,observed_difference
0,oc vs bagging,bio_sim_300,bagging-svm vs occ-svm,0.006199,0.056629
1,oc vs bagging,bio_sim_150,bagging-svm vs occ-svm,0.002600,0.062963
2,oc vs bagging,top_recall_150,bagging-svm vs occ-svm,0.065993,0.100023
3,oc vs bagging,top_recall_300,bagging-svm vs occ-svm,0.109489,0.117320
4,oc vs bagging,bedroc_150,bagging-svm vs occ-svm,0.050495,0.077942
5,oc vs bagging,bedroc_300,bagging-svm vs occ-svm,0.037296,0.098396
6,oc vs bagging,auroc,bagging-svm vs occ-svm,0.031697,0.086542
7,oc vs bagging,bio_sim_300,bagging-dnn vs occ-deepsvdd,0.019398,0.043757
8,oc vs bagging,bio_sim_150,bagging-dnn vs occ-deepsvdd,0.055194,0.033300
9,oc vs bagging,top_recall_150,bagging-dnn vs occ-deepsvdd,0.391261,-0.032446


In [9]:
results = []

for c in subresults['category'].unique():
    c_df = subresults[subresults['category'] == c].copy()

    for pair in [['bagging-svm', 'occ-svm'], ['bagging-dnn', 'occ-deepsvdd']]:
        for metric in selected_metrics:
            vals_a = c_df[c_df['method'] == pair[0]][metric].dropna().values
            vals_b = c_df[c_df['method'] == pair[1]][metric].dropna().values

            p_value, effect_size = run_permutation_test(vals_a, vals_b)

            results.append({
                'category':c,
                'setting': 'oc vs bagging',
                'metric': metric,
                'compared_groups': f'{pair[0]} vs {pair[1]}',
                'p_value': p_value,
                'observed_difference': effect_size
            })

results_oc_c = pd.DataFrame(results)

## fusion

In [10]:
merged_df

,disease,setting,method,category,bio_sim_300,bio_sim_150,top_recall_150,succ_150,top_recall_300,succ_300,bedroc_150,bedroc_300,auroc
0,ICD10_C16,df_gnn_occsvm_pred,early_fused,Neoplasms(8),0.197614,0.220574,0.20,1,0.32,1,1.056337e-01,0.178621,0.861021
1,ICD10_C16,df_gnn_occsvm_pred,early_fused_ppi,Neoplasms(8),0.161484,0.170398,0.28,1,0.40,1,2.069869e-01,0.287726,0.890605
2,ICD10_C16,df_gnn_occsvm_pred,late_fused,Neoplasms(8),0.179952,0.186959,0.16,1,0.32,1,1.279304e-01,0.213899,0.886881
3,ICD10_C16,df_gnn_occsvm_pred,late_fused_ppi,Neoplasms(8),0.151239,0.170379,0.32,1,0.44,1,2.363762e-01,0.328977,0.894945
4,ICD10_C16,df_gnn_occsvm_pred,ppi_emb_n2v,Neoplasms(8),0.164926,0.178091,0.32,1,0.48,1,2.400080e-01,0.328343,0.884768
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2491,ICD10_G43,gcn,ppi_emb_dw,Diseases of the nervous system(7),0.012557,0.011029,0.00,0,0.00,0,1.666501e-09,0.000027,0.711949
2492,ICD10_G43,gcn,literature_emb,Diseases of the nervous system(7),0.015636,0.013979,0.00,0,0.00,0,6.627426e-08,0.000129,0.696530
2493,ICD10_G43,gcn,seq_emb_port,Diseases of the nervous system(7),0.009462,0.009233,0.00,0,0.00,0,2.106836e-10,0.000007,0.568633
2494,ICD10_G43,gcn,seq_emb_esm,Diseases of the nervous system(7),0.011678,0.008830,0.00,0,0.00,0,1.081541e-11,0.000002,0.638304


In [26]:
svm_features = [
    "early_fused", "late_fused", "mid_linear_fused", "mid_geo_fused"
]

svm_ppi = [
    "early_fused_ppi", "late_fused_ppi",
    "mid_linear_fused_ppi", "mid_geo_fused_ppi"
]

dnn_features = [
    "early_fused", "late_fused", "mid_fused"
]

dnn_ppi = [
    "early_fused_ppi", "late_fused_ppi", "mid_fused_ppi"
]

settings_dict = {
    "Kernel_fused": ("df_gnn_occsvm_pred", svm_features),
    "Kernel_fused_ppi": ("df_gnn_occsvm_pred", svm_ppi),
    "DNN_fused": ("2019_nn_all_pred", dnn_features),
    "DNN_fused_ppi": ("2019_nn_all_pred", dnn_ppi),
}

all_fusion = []

for model, (setting_name, methods) in settings_dict.items():
    filtered_df = merged_df.loc[
        (merged_df["setting"] == setting_name)
        & (merged_df["method"].isin(methods))
    ].copy()

    filtered_df["model"] = model
    all_fusion.append(filtered_df)

all_fusion = pd.concat(all_fusion, ignore_index=True)

avg_fusion = (
    all_fusion
    .groupby(["model", "method", "category"], as_index=False)[selected_metrics]
    .mean()
)

avg_fusion = avg_fusion[
    ["model", "method", "category"] + selected_metrics
]

In [28]:
avg_fusion.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/mean_fusion_category.csv')

In [16]:
svm_features = ['early_fused', 'late_fused','mid_linear_fused','mid_geo_fused']
svm_ppi = ['early_fused_ppi', 'late_fused_ppi', 'mid_linear_fused_ppi', 'mid_geo_fused_ppi']
dnn_features = ['early_fused', 'late_fused', 'mid_fused']
dnn_ppi = ['early_fused_ppi', 'late_fused_ppi','mid_fused_ppi']

settings_dict = {'Kernel_fused':['df_gnn_occsvm_pred',svm_features],
            'Kernel_fused_ppi':['df_gnn_occsvm_pred',svm_ppi],
            'DNN_fused':['2019_nn_all_pred',dnn_features],
            'DNN_fused_ppi':['2019_nn_all_pred',dnn_ppi]}
results = []
for setting in list(settings_dict.keys()):
    for metric in selected_metrics:
        filtered_df = merged_df[(merged_df['setting']==settings_dict[setting][0])&(merged_df['method'].isin(settings_dict[setting][1]))]
        models = settings_dict[setting][1]
        for model_a, model_b in combinations(models, 2):
            vals_a = filtered_df[filtered_df['method'] == model_a][metric].dropna().values
            vals_b = filtered_df[filtered_df['method'] == model_b][metric].dropna().values
            p_value, effect_size = run_permutation_test(vals_a, vals_b)

            results.append({
                'setting':         setting,
                'metric':          metric,
                'compared_groups': f'{model_a} vs {model_b}',
                'p_value':         p_value,
                'effect_size':     effect_size
            })
fusion_all = pd.DataFrame(results)

In [18]:

results = []
for c in merged_df['category'].unique():
    c_df = merged_df[merged_df['category'] == c].copy()

    for setting in list(settings_dict.keys()):
        for metric in selected_metrics:
            filtered_df = c_df[(c_df['setting']==settings_dict[setting][0])&(c_df['method'].isin(settings_dict[setting][1]))]
            models = settings_dict[setting][1]
            for model_a, model_b in combinations(models, 2):
                vals_a = filtered_df[filtered_df['method'] == model_a][metric].dropna().values
                vals_b = filtered_df[filtered_df['method'] == model_b][metric].dropna().values
                p_value, effect_size = run_permutation_test(vals_a, vals_b)

                results.append({
                    'category': c,
                    'setting':         setting,
                    'metric':          metric,
                    'compared_groups': f'{model_a} vs {model_b}',
                    'p_value':         p_value,
                    'effect_size':     effect_size
                })
fusion_all_c = pd.DataFrame(results)

In [19]:
fusion_all_c

,category,setting,metric,compared_groups,p_value,effect_size
0,Neoplasms(8),Kernel_fused,bio_sim_300,early_fused vs late_fused,0.703430,0.015055
1,Neoplasms(8),Kernel_fused,bio_sim_300,early_fused vs mid_linear_fused,0.430157,0.031559
2,Neoplasms(8),Kernel_fused,bio_sim_300,early_fused vs mid_geo_fused,0.704130,0.016689
3,Neoplasms(8),Kernel_fused,bio_sim_300,late_fused vs mid_linear_fused,0.709829,0.016504
4,Neoplasms(8),Kernel_fused,bio_sim_300,late_fused vs mid_geo_fused,0.969703,0.001634
...,...,...,...,...,...,...
1381,Diseases of the skin and subcutaneous tissue(2),DNN_fused_ppi,bedroc_300,early_fused_ppi vs mid_fused_ppi,0.665333,-0.122365
1382,Diseases of the skin and subcutaneous tissue(2),DNN_fused_ppi,bedroc_300,late_fused_ppi vs mid_fused_ppi,0.663034,-0.120322
1383,Diseases of the skin and subcutaneous tissue(2),DNN_fused_ppi,auroc,early_fused_ppi vs late_fused_ppi,1.000000,0.007126
1384,Diseases of the skin and subcutaneous tissue(2),DNN_fused_ppi,auroc,early_fused_ppi vs mid_fused_ppi,0.664934,-0.010728


In [29]:
import pandas as pd
from pathlib import Path

# Input CSV files
csv_files = [
    '/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/permutation_mean_ocvsbag(across).csv',
    '/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/permutation_mean_ocvsbag_category.csv',
    '/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/permutation_mean_fusion(across).csv',
    '/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/permutation_mean_fusion_category.csv',
    '/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/mean_fusion_category.csv',
    '/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/mean_occvsbag_category.csv'
]

# Output Excel file
output_excel = "/itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/permutation_mean_results_summary_oc_fusion.xlsx"

# Description sheet content
description_lines = [
    "File Description",
    "",
    "1. P-values are calculated using a permutation test.",
    "2. Effect size is calculated using difference of means.",
    "3. Files with '_category' in the name are calculated using category groups.",
    "4. Files without '_category' are calculated by treating all samples as the same level.",
]

# Write to Excel
with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    # First sheet: description
    description_df = pd.DataFrame({"Description": description_lines})
    description_df.to_excel(writer, sheet_name="file_description", index=False)

    # Other sheets: one per CSV file, using CSV filename as sheet name
    for csv_path in csv_files:
        csv_path = Path(csv_path)
        sheet_name = csv_path.stem[:31]  # Excel sheet names max length = 31
        df = pd.read_csv(csv_path)
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Excel file saved to: {output_excel}")

Excel file saved to: /itf-fi-ml/shared/users/ziyuzh/svm/results/statistic/permutation_mean_results_summary_oc_fusion.xlsx


In [3]:
fusion_features = ['early_fused', 'late_fused','mid_linear_fused','mid_geo_fused','early_fused_ppi', 'late_fused_ppi', 'mid_linear_fused_ppi', 'mid_geo_fused_ppi', 'mid_fused','mid_fused_ppi']
fusion_df = merged_df[merged_df['method'].isin(fusion_features)]

In [4]:
fusion_df_mean_across_method = (
    fusion_df
    .groupby(['setting', 'category'], as_index=False)
    .mean(numeric_only=True)
)

In [6]:
fusion_df_mean_across_method['setting'].unique()

array(['2019_nn_all_pred', 'df_gnn_occsvm_pred'], dtype=object)

In [9]:
fusion_df_mean_across_method[fusion_df_mean_across_method['setting']=='df_gnn_occsvm_pred'].round(3)

,setting,category,bio_sim_300,bio_sim_150,top_recall_150,succ_150,top_recall_300,succ_300,bedroc_150,bedroc_300,auroc
11,df_gnn_occsvm_pred,Diseases of the blood and certain disorders(3),0.062,0.076,0.375,0.375,0.500,0.500,0.284,0.376,0.913
12,df_gnn_occsvm_pred,Diseases of the circulatory system(7),0.189,0.199,0.208,0.804,0.254,0.857,0.158,0.199,0.851
13,df_gnn_occsvm_pred,Diseases of the digestive system(2),0.245,0.269,0.031,0.125,0.203,0.500,0.028,0.085,0.827
14,df_gnn_occsvm_pred,Diseases of the genitourinary system(6),0.177,0.196,0.181,0.312,0.208,0.417,0.091,0.144,0.756
15,df_gnn_occsvm_pred,Diseases of the musculoskeletal system and con...,0.142,0.161,0.312,0.417,0.417,0.500,0.184,0.291,0.916
16,df_gnn_occsvm_pred,Diseases of the nervous system(7),0.120,0.126,0.109,0.161,0.192,0.321,0.060,0.109,0.737
17,df_gnn_occsvm_pred,Diseases of the respiratory system(3),0.157,0.163,0.333,0.333,0.611,0.667,0.221,0.353,0.958
18,df_gnn_occsvm_pred,Diseases of the skin and subcutaneous tissue(2),0.248,0.266,0.500,0.500,0.500,0.500,0.456,0.489,0.965
19,df_gnn_occsvm_pred,"Endocrine, nutritional and metabolic diseases(2)",0.297,0.265,0.277,0.500,0.339,0.500,0.188,0.246,0.757
20,df_gnn_occsvm_pred,Mental and behavioural disorders(5),0.133,0.114,0.034,0.250,0.056,0.400,0.018,0.034,0.669


In [10]:
fusion_df_mean_across_method[fusion_df_mean_across_method['setting']=='2019_nn_all_pred'].round(3)

,setting,category,bio_sim_300,bio_sim_150,top_recall_150,succ_150,top_recall_300,succ_300,bedroc_150,bedroc_300,auroc
0,2019_nn_all_pred,Diseases of the blood and certain disorders(3),0.071,0.083,0.222,0.222,0.556,0.556,0.120,0.252,0.819
1,2019_nn_all_pred,Diseases of the circulatory system(7),0.171,0.159,0.207,0.738,0.245,0.833,0.152,0.193,0.831
2,2019_nn_all_pred,Diseases of the digestive system(2),0.247,0.263,0.021,0.167,0.146,0.417,0.025,0.074,0.809
3,2019_nn_all_pred,Diseases of the genitourinary system(6),0.170,0.175,0.157,0.306,0.222,0.417,0.114,0.156,0.746
4,2019_nn_all_pred,Diseases of the musculoskeletal system and con...,0.144,0.156,0.167,0.222,0.194,0.278,0.108,0.174,0.899
5,2019_nn_all_pred,Diseases of the nervous system(7),0.108,0.088,0.167,0.262,0.244,0.381,0.088,0.142,0.735
6,2019_nn_all_pred,Diseases of the respiratory system(3),0.152,0.147,0.444,0.444,0.500,0.500,0.247,0.357,0.942
7,2019_nn_all_pred,Diseases of the skin and subcutaneous tissue(2),0.257,0.271,0.500,0.500,0.542,0.583,0.468,0.516,0.959
8,2019_nn_all_pred,"Endocrine, nutritional and metabolic diseases(2)",0.187,0.136,0.071,0.333,0.179,0.500,0.044,0.097,0.714
9,2019_nn_all_pred,Mental and behavioural disorders(5),0.056,0.040,0.039,0.333,0.053,0.400,0.022,0.036,0.604


In [11]:
fusion_df_mean = (
    fusion_df
    .groupby(['setting', 'method', 'category'], as_index=False)
    .mean(numeric_only=True)
)

In [14]:
fusion_df_mean.groupby('category', group_keys=False).apply(lambda df: df.nlargest(2, 'top_recall_150'))

/tmp/ipykernel_1694258/3166412521.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fusion_df_mean.groupby('category', group_keys=False).apply(lambda df: df.nlargest(2, 'top_recall_150'))


,setting,method,category,bio_sim_300,bio_sim_150,top_recall_150,succ_150,top_recall_300,succ_300,bedroc_150,bedroc_300,auroc
121,df_gnn_occsvm_pred,mid_geo_fused_ppi,Diseases of the blood and certain disorders(3),0.049982,0.058586,0.666667,0.666667,0.666667,0.666667,0.304791,0.436848,0.946116
11,2019_nn_all_pred,early_fused_ppi,Diseases of the blood and certain disorders(3),0.070280,0.091580,0.333333,0.333333,0.666667,0.666667,0.171603,0.328398,0.870620
45,2019_nn_all_pred,mid_fused,Diseases of the circulatory system(7),0.164323,0.165873,0.237118,0.857143,0.276679,0.857143,0.187090,0.228807,0.867789
56,2019_nn_all_pred,mid_fused_ppi,Diseases of the circulatory system(7),0.170732,0.166571,0.231624,0.857143,0.263346,0.857143,0.170518,0.218658,0.871285
90,df_gnn_occsvm_pred,late_fused,Diseases of the digestive system(2),0.260928,0.281237,0.125000,0.500000,0.187500,0.500000,0.055795,0.117669,0.847406
134,df_gnn_occsvm_pred,mid_linear_fused,Diseases of the digestive system(2),0.246857,0.278419,0.125000,0.500000,0.250000,0.500000,0.039909,0.100730,0.857970
47,2019_nn_all_pred,mid_fused,Diseases of the genitourinary system(6),0.174464,0.184825,0.277778,0.500000,0.277778,0.500000,0.171328,0.217290,0.752023
25,2019_nn_all_pred,late_fused,Diseases of the genitourinary system(6),0.164351,0.178272,0.194444,0.333333,0.194444,0.333333,0.129413,0.172081,0.740336
125,df_gnn_occsvm_pred,mid_geo_fused_ppi,Diseases of the musculoskeletal system and con...,0.119342,0.138255,0.666667,0.666667,0.666667,0.666667,0.305617,0.439528,0.944677
114,df_gnn_occsvm_pred,mid_geo_fused,Diseases of the musculoskeletal system and con...,0.147991,0.177852,0.500000,0.666667,0.500000,0.666667,0.286984,0.370947,0.936056


In [13]:
fusion_df_mean[
    fusion_df_mean['method'].str.contains('early')
].sort_values(
    by=['top_recall_150', 'top_recall_300', 'bedroc_150', 'bedroc_300'],
    ascending=False
)

,setting,method,category,bio_sim_300,bio_sim_150,top_recall_150,succ_150,top_recall_300,succ_300,bedroc_150,bedroc_300,auroc
17,2019_nn_all_pred,early_fused_ppi,Diseases of the respiratory system(3),0.156435,0.141529,0.666667,0.666667,0.666667,0.666667,0.416791,0.527390,0.951933
83,df_gnn_occsvm_pred,early_fused_ppi,Diseases of the respiratory system(3),0.150760,0.161893,0.666667,0.666667,0.666667,0.666667,0.326947,0.461309,0.970485
84,df_gnn_occsvm_pred,early_fused_ppi,Diseases of the skin and subcutaneous tissue(2),0.251057,0.282601,0.500000,0.500000,0.500000,0.500000,0.480384,0.502986,0.974428
18,2019_nn_all_pred,early_fused_ppi,Diseases of the skin and subcutaneous tissue(2),0.267916,0.272600,0.500000,0.500000,0.500000,0.500000,0.433875,0.486436,0.979585
7,2019_nn_all_pred,early_fused,Diseases of the skin and subcutaneous tissue(2),0.244119,0.237989,0.500000,0.500000,0.500000,0.500000,0.428454,0.462849,0.852239
73,df_gnn_occsvm_pred,early_fused,Diseases of the skin and subcutaneous tissue(2),0.247983,0.235253,0.500000,0.500000,0.500000,0.500000,0.394581,0.444176,0.916240
11,2019_nn_all_pred,early_fused_ppi,Diseases of the blood and certain disorders(3),0.070280,0.091580,0.333333,0.333333,0.666667,0.666667,0.171603,0.328398,0.870620
77,df_gnn_occsvm_pred,early_fused_ppi,Diseases of the blood and certain disorders(3),0.064998,0.085690,0.333333,0.333333,0.333333,0.333333,0.245717,0.333791,0.940221
66,df_gnn_occsvm_pred,early_fused,Diseases of the blood and certain disorders(3),0.073762,0.082035,0.333333,0.333333,0.333333,0.333333,0.185671,0.250282,0.790964
85,df_gnn_occsvm_pred,early_fused_ppi,"Endocrine, nutritional and metabolic diseases(2)",0.291750,0.277416,0.285714,0.500000,0.357143,0.500000,0.164998,0.235315,0.764180


In [9]:
fusion_df_mean['category'].unique()

array(['Diseases of the blood and certain disorders(3)',
       'Diseases of the circulatory system(7)',
       'Diseases of the digestive system(2)',
       'Diseases of the genitourinary system(6)',
       'Diseases of the musculoskeletal system and connective tissue(3)',
       'Diseases of the nervous system(7)',
       'Diseases of the respiratory system(3)',
       'Diseases of the skin and subcutaneous tissue(2)',
       'Endocrine, nutritional and metabolic diseases(2)',
       'Mental and behavioural disorders(5)', 'Neoplasms(8)'],
      dtype=object)

In [10]:
fusion_df_mean[fusion_df_mean['category'] == 'Diseases of the blood and certain disorders(3)'].round(3)

,setting,method,category,bio_sim_300,bio_sim_150,top_recall_150,succ_150,top_recall_300,succ_300,bedroc_150,bedroc_300,auroc
0,2019_nn_all_pred,early_fused,Diseases of the blood and certain disorders(3),0.078,0.081,0.000,0.000,0.000,0.000,0.001,0.015,0.625
11,2019_nn_all_pred,early_fused_ppi,Diseases of the blood and certain disorders(3),0.070,0.092,0.333,0.333,0.667,0.667,0.172,0.328,0.871
22,2019_nn_all_pred,late_fused,Diseases of the blood and certain disorders(3),0.072,0.083,0.000,0.000,0.667,0.667,0.052,0.186,0.861
33,2019_nn_all_pred,late_fused_ppi,Diseases of the blood and certain disorders(3),0.073,0.092,0.333,0.333,0.667,0.667,0.145,0.310,0.855
44,2019_nn_all_pred,mid_fused,Diseases of the blood and certain disorders(3),0.065,0.077,0.333,0.333,0.667,0.667,0.151,0.315,0.851
55,2019_nn_all_pred,mid_fused_ppi,Diseases of the blood and certain disorders(3),0.068,0.074,0.333,0.333,0.667,0.667,0.202,0.357,0.854
66,df_gnn_occsvm_pred,early_fused,Diseases of the blood and certain disorders(3),0.074,0.082,0.333,0.333,0.333,0.333,0.186,0.250,0.791
77,df_gnn_occsvm_pred,early_fused_ppi,Diseases of the blood and certain disorders(3),0.065,0.086,0.333,0.333,0.333,0.333,0.246,0.334,0.940
88,df_gnn_occsvm_pred,late_fused,Diseases of the blood and certain disorders(3),0.068,0.086,0.333,0.333,0.333,0.333,0.320,0.379,0.922
99,df_gnn_occsvm_pred,late_fused_ppi,Diseases of the blood and certain disorders(3),0.057,0.073,0.333,0.333,0.667,0.667,0.299,0.413,0.948


### no category

In [10]:
svm_features = ['early_fused', 'late_fused','mid_linear_fused','mid_geo_fused']
svm_ppi = ['early_fused_ppi', 'late_fused_ppi', 'mid_linear_fused_ppi', 'mid_geo_fused_ppi']
dnn_features = ['early_fused', 'late_fused', 'mid_fused']
dnn_ppi = ['early_fused_ppi', 'late_fused_ppi','mid_fused_ppi']

settings_dict = {'Kernel_fused':['df_gnn_occsvm_pred',svm_features],
            'Kernel_fused_ppi':['df_gnn_occsvm_pred',svm_ppi],
            'DNN_fused':['2019_nn_all_pred',dnn_features],
            'DNN_fused_ppi':['2019_nn_all_pred',dnn_ppi]}

for setting in list(settings_dict.keys()):
    for metric in selected_metrics:
        filtered_df = merged_df[(merged_df['setting']==settings_dict[setting][0])&(merged_df['method'].isin(settings_dict[setting][1]))]
        break
    break

In [11]:
filtered_df

,disease,setting,method,category,bio_sim_300,bio_sim_150,top_recall_150,succ_150,top_recall_300,succ_300,bedroc_150,bedroc_300,auroc
0,ICD10_C16,df_gnn_occsvm_pred,early_fused,Neoplasms(8),0.197614,0.220574,0.20,1,0.32,1,1.056337e-01,1.786213e-01,0.861021
2,ICD10_C16,df_gnn_occsvm_pred,late_fused,Neoplasms(8),0.179952,0.186959,0.16,1,0.32,1,1.279304e-01,2.138990e-01,0.886881
10,ICD10_C16,df_gnn_occsvm_pred,mid_linear_fused,Neoplasms(8),0.155381,0.172914,0.20,1,0.36,1,1.962768e-01,2.776490e-01,0.895194
11,ICD10_C16,df_gnn_occsvm_pred,mid_geo_fused,Neoplasms(8),0.163062,0.199242,0.08,1,0.16,1,7.314478e-02,1.431191e-01,0.888830
34,ICD10_F31,df_gnn_occsvm_pred,early_fused,Mental and behavioural disorders(5),0.304136,0.230836,0.00,0,0.00,0,1.833038e-33,4.281399e-17,0.531484
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1575,ICD10_K44,df_gnn_occsvm_pred,mid_geo_fused,Diseases of the digestive system(2),0.200726,0.237480,0.00,0,0.00,0,6.662824e-08,1.290845e-04,0.793501
1598,ICD10_G43,df_gnn_occsvm_pred,early_fused,Diseases of the nervous system(7),0.034221,0.043478,0.00,0,0.25,1,2.394231e-02,7.971000e-02,0.757438
1600,ICD10_G43,df_gnn_occsvm_pred,late_fused,Diseases of the nervous system(7),0.022201,0.039773,0.00,0,0.25,1,1.359704e-02,5.833523e-02,0.860850
1608,ICD10_G43,df_gnn_occsvm_pred,mid_linear_fused,Diseases of the nervous system(7),0.020374,0.033742,0.00,0,0.00,0,5.178619e-03,3.627345e-02,0.836334


In [12]:
svm_features = ['early_fused', 'late_fused','mid_linear_fused','mid_geo_fused']
svm_ppi = ['early_fused_ppi', 'late_fused_ppi', 'mid_linear_fused_ppi', 'mid_geo_fused_ppi']
dnn_features = ['early_fused', 'late_fused', 'mid_fused']
dnn_ppi = ['early_fused_ppi', 'late_fused_ppi','mid_fused_ppi']

settings_dict = {'Kernel_fused':['df_gnn_occsvm_pred',svm_features],
            'Kernel_fused_ppi':['df_gnn_occsvm_pred',svm_ppi],
            'DNN_fused':['2019_nn_all_pred',dnn_features],
            'DNN_fused_ppi':['2019_nn_all_pred',dnn_ppi]}
results = []
for setting in list(settings_dict.keys()):
    filtered_df = merged_df[(merged_df['setting']==settings_dict[setting][0])&(merged_df['method'].isin(settings_dict[setting][1]))]
    for metric in selected_metrics:
        mean_scores = (
            filtered_df
            .groupby('method')[metric]
            .mean()
            .dropna()
        )
        best_method = mean_scores.idxmax()

        others = [m for m in settings_dict[setting][1] if m != best_method]

        for m in others:
            
            vals_a = filtered_df[filtered_df['method'] == best_method][metric].dropna().values
            vals_b = filtered_df[filtered_df['method'] == m][metric].dropna().values

            p_value, effect_size = run_permutation_test(vals_a, vals_b)

            results.append({
                'setting': setting,
                'metric': metric,
                'compared_groups': f'{best_method} vs {m}',
                'p_value': p_value,
                'observed_difference': effect_size
            })

results_fusion = pd.DataFrame(results)


In [13]:
results_fusion

,setting,metric,compared_groups,p_value,observed_difference
0,Kernel_fused,bio_sim_300,late_fused vs early_fused,0.845515,0.004301
1,Kernel_fused,bio_sim_300,late_fused vs mid_linear_fused,0.698730,0.009339
2,Kernel_fused,bio_sim_300,late_fused vs mid_geo_fused,0.870213,0.004167
3,Kernel_fused,bio_sim_150,mid_geo_fused vs early_fused,0.414259,0.017004
4,Kernel_fused,bio_sim_150,mid_geo_fused vs late_fused,0.948605,0.001384
...,...,...,...,...,...
65,DNN_fused_ppi,bedroc_150,mid_fused_ppi vs late_fused_ppi,0.316768,0.043534
66,DNN_fused_ppi,bedroc_300,mid_fused_ppi vs early_fused_ppi,0.589441,0.028491
67,DNN_fused_ppi,bedroc_300,mid_fused_ppi vs late_fused_ppi,0.403760,0.042452
68,DNN_fused_ppi,auroc,early_fused_ppi vs late_fused_ppi,0.956704,0.001880


## categories

In [4]:
import pandas as pd

df = pd.read_excel("/itf-fi-ml/shared/users/ziyuzh/svm/results/1and2percent_results/150_300_mean_results.xlsx", sheet_name="fusion_df_mean")

In [13]:
df

,setting,method,category,bio_sim_300,bio_sim_150,top_recall_150,succ_150,top_recall_300,succ_300,bedroc_150,bedroc_300,auroc,feature
0,2019_nn_all_pred,DNN,Diseases of the blood and certain disorders(3),0.077629,0.081067,0.000000,0.000000,0.000000,0.000000,0.000704,0.015320,0.625233,early_fused
1,2019_nn_all_pred,DNN,Diseases of the circulatory system(7),0.150928,0.123431,0.111111,0.285714,0.199243,0.714286,0.103787,0.140116,0.687584,early_fused
2,2019_nn_all_pred,DNN,Diseases of the digestive system(2),0.246181,0.250304,0.000000,0.000000,0.062500,0.500000,0.008770,0.050933,0.785038,early_fused
3,2019_nn_all_pred,DNN,Diseases of the genitourinary system(6),0.160091,0.149537,0.000000,0.000000,0.107143,0.333333,0.012148,0.037368,0.729003,early_fused
4,2019_nn_all_pred,DNN,Diseases of the musculoskeletal system and con...,0.148658,0.152922,0.000000,0.000000,0.000000,0.000000,0.007470,0.042055,0.748872,early_fused
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,df_gnn_occsvm_pred,SVM,Diseases of the respiratory system(3),0.145547,0.151389,0.333333,0.333333,0.666667,0.666667,0.291868,0.437009,0.972374,mid_linear_fused_ppi
150,df_gnn_occsvm_pred,SVM,Diseases of the skin and subcutaneous tissue(2),0.245369,0.266477,0.500000,0.500000,0.500000,0.500000,0.481441,0.513520,0.979152,mid_linear_fused_ppi
151,df_gnn_occsvm_pred,SVM,"Endocrine, nutritional and metabolic diseases(2)",0.302259,0.271549,0.285714,0.500000,0.357143,0.500000,0.188789,0.253298,0.755004,mid_linear_fused_ppi
152,df_gnn_occsvm_pred,SVM,Mental and behavioural disorders(5),0.135208,0.125832,0.047143,0.400000,0.067143,0.400000,0.018432,0.038185,0.669924,mid_linear_fused_ppi


In [37]:
import pandas as pd

# metrics
perf_metrics = [
    "top_recall_150", "succ_150",
    "top_recall_300", "succ_300",
    "bedroc_150", "bedroc_300"
]

bio_metrics = ["bio_sim_300", "bio_sim_150"]

methods = ["DNN", "SVM"]

# keep features: early/mid/late, exclude ppi
df_sub = df[
    df["method"].isin(methods)
    & ~df["feature"].str.contains("ppi", case=False, na=False)
    & df["feature"].str.contains(r"^(early|mid|late)", case=False, na=False)
].copy()


def get_best_features(group, metrics, min_better):
    """
    For each feature, count how many metrics are equal to the best value
    within this category-method group.
    Return features with at least min_better best metrics.
    """
    best_values = group[metrics].max(numeric_only=True)

    score_df = group[["category", "method", "feature"] + metrics].copy()

    for m in metrics:
        score_df[f"{m}_is_best"] = score_df[m] == best_values[m]

    best_cols = [f"{m}_is_best" for m in metrics]
    score_df["n_best_metrics"] = score_df[best_cols].sum(axis=1)

    return score_df[score_df["n_best_metrics"] >= min_better].copy()


results = []

for (category, method), group in df_sub.groupby(["category", "method"]):
    perf_best = get_best_features(group, perf_metrics, min_better=3)
    bio_best = get_best_features(group, bio_metrics, min_better=1)

    overlap = perf_best.merge(
        bio_best,
        on=["category", "method", "feature"],
        suffixes=("_perf", "_bio")
    )

    results.append({
        "category": category,
        "method": method,
        "perf_best_features": perf_best["feature"].tolist(),
        "bio_best_features": bio_best["feature"].tolist(),
        "overlap_features": overlap["feature"].tolist()
    })

summary = pd.DataFrame(results)

/tmp/ipykernel_2192808/2270767713.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  & df["feature"].str.contains(r"^(early|mid|late)", case=False, na=False)


In [38]:
summary

,category,method,perf_best_features,bio_best_features,overlap_features
0,Diseases of the blood and certain disorders(3),DNN,[mid_fused],"[early_fused, late_fused]",[]
1,Diseases of the blood and certain disorders(3),SVM,"[late_fused, mid_linear_fused]","[early_fused, late_fused]",[late_fused]
2,Diseases of the circulatory system(7),DNN,[mid_fused],"[late_fused, mid_fused]",[mid_fused]
3,Diseases of the circulatory system(7),SVM,"[early_fused, mid_linear_fused]",[late_fused],[]
4,Diseases of the digestive system(2),DNN,[mid_fused],[late_fused],[]
5,Diseases of the digestive system(2),SVM,"[late_fused, mid_linear_fused]","[late_fused, mid_geo_fused]",[late_fused]
6,Diseases of the genitourinary system(6),DNN,[mid_fused],[mid_fused],[mid_fused]
7,Diseases of the genitourinary system(6),SVM,"[late_fused, mid_geo_fused, mid_linear_fused]",[mid_linear_fused],[mid_linear_fused]
8,Diseases of the musculoskeletal system and con...,DNN,[mid_fused],"[late_fused, mid_fused]",[mid_fused]
9,Diseases of the musculoskeletal system and con...,SVM,[mid_geo_fused],"[early_fused, mid_geo_fused]",[mid_geo_fused]


In [39]:
overlap_rows = []

for (category, method), group in df_sub.groupby(["category", "method"]):
    perf_best = get_best_features(group, perf_metrics, min_better=3)
    bio_best = get_best_features(group, bio_metrics, min_better=1)

    overlap_features = set(perf_best["feature"]) & set(bio_best["feature"])

    for feature in overlap_features:
        overlap_rows.append({
            "category": category,
            "method": method,
            "feature": feature
        })

overlap_df = pd.DataFrame(overlap_rows)

In [40]:
overlap_df

,category,method,feature
0,Diseases of the blood and certain disorders(3),SVM,late_fused
1,Diseases of the circulatory system(7),DNN,mid_fused
2,Diseases of the digestive system(2),SVM,late_fused
3,Diseases of the genitourinary system(6),DNN,mid_fused
4,Diseases of the genitourinary system(6),SVM,mid_linear_fused
5,Diseases of the musculoskeletal system and con...,DNN,mid_fused
6,Diseases of the musculoskeletal system and con...,SVM,mid_geo_fused
7,Diseases of the nervous system(7),DNN,late_fused
8,Diseases of the skin and subcutaneous tissue(2),DNN,late_fused
9,Diseases of the skin and subcutaneous tissue(2),DNN,mid_fused


In [17]:
svm_mid_better

,method,category,target_stage,better_than_early,better_than_late
0,SVM,Diseases of the blood and certain disorders(3),mid,4,3
1,SVM,Diseases of the musculoskeletal system and con...,mid,6,6
2,SVM,Diseases of the nervous system(7),mid,5,3
3,SVM,"Endocrine, nutritional and metabolic diseases(2)",mid,4,3
4,SVM,Mental and behavioural disorders(5),mid,3,3
5,SVM,Neoplasms(8),mid,6,4


In [21]:
dnn_mid_better

,method,category,target_stage,better_than_early,better_than_late
0,DNN,Diseases of the blood and certain disorders(3),mid,6,4
1,DNN,Diseases of the circulatory system(7),mid,6,4
2,DNN,Diseases of the digestive system(2),mid,5,6
3,DNN,Diseases of the genitourinary system(6),mid,6,6
4,DNN,Diseases of the musculoskeletal system and con...,mid,6,5
5,DNN,Neoplasms(8),mid,6,6


In [22]:
dnn_late_better

,method,category,target_stage,better_than_early,better_than_mid
0,DNN,Diseases of the nervous system(7),late,6,3


In [23]:
svm_late_better

,method,category,target_stage,better_than_early,better_than_mid
0,SVM,Diseases of the digestive system(2),late,4,4
1,SVM,Diseases of the nervous system(7),late,3,3


In [24]:
biosim_results = {}

for method in methods:
    for stage in stages:
        key = f"{method}_{stage}_biosim"
        biosim_results[key] = find_best_stage_categories(
            df=df,
            method=method,
            target_stage=stage,
            metrics=metrics_biosim,
            min_better=1,   # use 2 if both biosim metrics must be better
            exclude_ppi=True,
        )

In [26]:
dnn_early_bio = biosim_results["DNN_early_biosim"]
dnn_mid_bio = biosim_results["DNN_mid_biosim"]
dnn_late_bio = biosim_results["DNN_late_biosim"]

svm_early_bio = biosim_results["SVM_early_biosim"]
svm_mid_bio = biosim_results["SVM_mid_biosim"]
svm_late_bio = biosim_results["SVM_late_biosim"]

In [33]:
from collections import Counter

def find_overlap_categories(dfs, min_count=3):
    """
    dfs: list of DataFrames (each must have 'category')
    min_count: minimum number of appearances
    """
    counter = Counter()

    for df in dfs:
        if df is None or df.empty:
            continue
        cats = df["category"].dropna().unique()
        counter.update(cats)

    overlap = [cat for cat, cnt in counter.items() if cnt >= min_count]

    return overlap, counter

In [34]:
early_dfs = [
    dnn_early_bio,
    svm_early_bio,
    dnn_early_better,
    svm_early_better,
]

early_overlap, early_counts = find_overlap_categories(early_dfs, min_count=3)

print("Early overlap categories:", early_overlap)

Early overlap categories: []


In [35]:
mid_dfs = [
    dnn_mid_bio,
    svm_mid_bio,
    dnn_mid_better,
    svm_mid_better,
]

mid_overlap, mid_counts = find_overlap_categories(mid_dfs, min_count=3)

print("Mid overlap categories:", mid_overlap)

Mid overlap categories: ['Diseases of the genitourinary system(6)', 'Diseases of the musculoskeletal system and connective tissue(3)']


In [36]:
late_dfs = [
    dnn_late_bio,
    svm_late_bio,
    dnn_late_better,
    svm_late_better,
]

late_overlap, late_counts = find_overlap_categories(late_dfs, min_count=3)

print("Late overlap categories:", late_overlap)

Late overlap categories: ['Diseases of the digestive system(2)', 'Diseases of the nervous system(7)']


In [27]:
dnn_early_bio

,method,category,target_stage,better_than_mid,better_than_late
0,DNN,Diseases of the blood and certain disorders(3),early,2,1
1,DNN,Diseases of the respiratory system(3),early,2,2
2,DNN,Mental and behavioural disorders(5),early,2,2


In [28]:
svm_early_bio

,method,category,target_stage,better_than_mid,better_than_late
0,SVM,Diseases of the blood and certain disorders(3),early,2,1
1,SVM,Diseases of the genitourinary system(6),early,1,1
2,SVM,Diseases of the musculoskeletal system and con...,early,1,1
3,SVM,Diseases of the respiratory system(3),early,2,2
4,SVM,Neoplasms(8),early,2,2


In [29]:
dnn_mid_bio

,method,category,target_stage,better_than_early,better_than_late
0,DNN,Diseases of the circulatory system(7),mid,2,1
1,DNN,Diseases of the genitourinary system(6),mid,2,2
2,DNN,Diseases of the musculoskeletal system and con...,mid,1,1
3,DNN,Diseases of the skin and subcutaneous tissue(2),mid,2,1


In [30]:
svm_mid_bio

,method,category,target_stage,better_than_early,better_than_late
0,SVM,Diseases of the digestive system(2),mid,1,1
1,SVM,Diseases of the genitourinary system(6),mid,1,1
2,SVM,Diseases of the skin and subcutaneous tissue(2),mid,2,1
3,SVM,"Endocrine, nutritional and metabolic diseases(2)",mid,2,1
4,SVM,Mental and behavioural disorders(5),mid,2,2


In [31]:
dnn_late_bio

,method,category,target_stage,better_than_early,better_than_mid
0,DNN,Diseases of the blood and certain disorders(3),late,1,2
1,DNN,Diseases of the circulatory system(7),late,2,1
2,DNN,Diseases of the digestive system(2),late,2,2
3,DNN,Diseases of the musculoskeletal system and con...,late,2,1
4,DNN,Diseases of the nervous system(7),late,2,2
5,DNN,Diseases of the skin and subcutaneous tissue(2),late,2,1
6,DNN,"Endocrine, nutritional and metabolic diseases(2)",late,2,2
7,DNN,Neoplasms(8),late,2,2


In [32]:
svm_late_bio

,method,category,target_stage,better_than_early,better_than_mid
0,SVM,Diseases of the blood and certain disorders(3),late,1,2
1,SVM,Diseases of the circulatory system(7),late,2,2
2,SVM,Diseases of the digestive system(2),late,2,1
3,SVM,Diseases of the genitourinary system(6),late,1,1
4,SVM,Diseases of the musculoskeletal system and con...,late,1,2
5,SVM,Diseases of the nervous system(7),late,2,2
6,SVM,Diseases of the skin and subcutaneous tissue(2),late,2,1
7,SVM,"Endocrine, nutritional and metabolic diseases(2)",late,2,1
